# Customer Churn Prediction — Exploratory Data Analysis (EDA)

**Author:** Gokul  
**Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`)  
**Objective:** Understand customer demographics, account characteristics, service usage, and identify key drivers of customer attrition.

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

## 2. Dataset Overview & Data Quality Audit

In [ ]:
print('--- Column Data Types & Non-Null Counts ---')
print(df.info())

print('\n--- Missing Values Per Column ---')
print(df.isnull().sum())

print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 3. Data Cleaning & Type Correction
Notice that `TotalCharges` is stored as an object/string. This occurs because new customers with `tenure = 0` have whitespace values (`' '`). We convert `TotalCharges` to numeric and inspect.

In [ ]:
# Convert TotalCharges with coercion
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
missing_tc = df['TotalCharges'].isnull().sum()
print(f'Number of missing TotalCharges entries: {missing_tc}')
display(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']])

# Drop these 11 rows (< 0.16% of dataset)
df = df.dropna(subset=['TotalCharges']).copy()
print(f'Cleaned dataset shape: {df.shape}')

## 4. Target Variable Analysis (`Churn`)
Let us inspect the distribution of the target variable to understand class imbalance.

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print(pd.DataFrame({'Count': churn_counts, 'Percentage (%)': churn_pct.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Customer Churn Class Distribution', fontsize=14)
ax.set_ylabel('Number of Customers')
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{height} ({height/len(df)*100:.1f}%)',
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

**Key Insight:** ~26.5% of customers have churned while 73.5% remained. This represents an imbalanced classification problem, meaning **Accuracy alone is misleading**; Precision, Recall, F1, and ROC-AUC are critical metrics.

## 5. Numerical Features Analysis
We examine distributions of `tenure`, `MonthlyCharges`, and `TotalCharges` across churn categories.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='Churn', y=col, palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'{col} by Churn Status', fontsize=12)
plt.tight_layout()
plt.show()

print(df.groupby('Churn')[num_cols].agg(['median', 'mean']).round(2))

## 6. Categorical Features & Service Breakdown
Let us examine how churn rates vary across contract types, internet services, and payment methods.

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, hue='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[idx])
    axes[idx].set_title(f'Churn Distribution by {col}', fontsize=13)
    axes[idx].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## 7. Key Findings & Recommendations for Modeling
1. **Contract Type:** Customers on Month-to-month contracts churn at dramatically higher rates (>42%) compared to One-year (~11%) or Two-year (~3%) contracts.
2. **Tenure:** Churn is heavily concentrated in the first 0-12 months of service. Long-term loyalty drastically reduces churn probability.
3. **Internet Service:** Fiber optic subscribers experience high churn, frequently driven by high MonthlyCharges and lack of bundled Tech Support.
4. **Payment Method:** Electronic check users exhibit the highest churn rate among all payment methods.
5. **Class Imbalance Strategy:** Use class weighting (`class_weight='balanced'`) in classifiers or calibrate decision thresholds to maintain high Recall.